# Your Deep Agent Streams. Wrap It in MCP and It Goes Silent.

*We expose a streaming Deep Agents agent as an MCP tool, watch the parent agent go dark for 34.6 seconds, and fix it with a small progress relay that keeps every child thought and tool update on screen.*

Picture the account-support desk at a large telecom. An agent on the floor types one request into the internal assistant: validate two customer emails and pull their profiles. The assistant thinks for two seconds, says it is calling a tool, and then shows nothing at all for more than half a minute.

Nothing is broken. Somewhere behind that tool call, a second agent is working hard, reasoning, calling four tools and streaming twelve progress updates. None of it reaches the screen.

I ran into exactly this while wiring a Deep Agents agent behind an MCP server for a parent orchestrator. The child streamed beautifully on its own. The moment it became somebody else's tool, the stream stopped at the MCP boundary.

## Agents are becoming tools for other agents

Teams are no longer shipping one agent. They ship a specialist agent, expose it as a tool over the [Model Context Protocol](https://modelcontextprotocol.io/specification/2025-06-18), and let a parent agent route work to it. LangChain's [Deep Agents](https://github.com/langchain-ai/deepagents) harness makes the specialist easy to build, and [FastMCP](https://gofastmcp.com/) makes it easy to serve.

The catch is that an MCP tool call is, by default, request in and result out. A child agent that takes 35 seconds to do its job is a 35-second blank in the parent's UI. For a support desk, a blank screen reads as a hang, and people click away or retry, which doubles the load.

The protocol already has the pipe we need. MCP [progress notifications](https://modelcontextprotocol.io/specification/2025-06-18/basic/utilities/progress) let a server push updates during a long tool call, each carrying a free-text `message`. What is missing is a convention for pushing a whole agent's event stream through that one string field.

Here is what we will do, in order:

1. Build a child MCP server with two slow, streaming tools.
2. Build a reusable Deep Agents runner that turns MCP progress into LangChain stream events.
3. Wrap the child agent in a parent MCP tool the naive way, and measure the silence.
4. Add a progress relay and measure again.
5. Make sure clients that ignore progress still get the full story.

## What is a progress relay?

A progress relay is an MCP tool that runs another agent and re-emits every event from that agent as an MCP progress notification, packed into a tagged JSON string so the caller can unpack it.

The arithmetic shows why we need one. Our request makes the child agent call four tools: validate and profile, for two addresses. Each tool reports three stages, and each stage takes 2 seconds in this demo. That is 4 × 3 × 2 = 24 seconds of tool time before we add a single model turn. In the companion repo the stage delay is 5 seconds, which makes it 4 × 3 × 5 = 60 seconds. Without a relay, all of that time is invisible to the parent's user.

Think of a parcel delivery. The courier hands the parcel to a local partner for the last mile. If the partner's scans never flow back to the original tracking page, the customer sees "out for delivery" for six hours and assumes it is lost. The relay is the integration that forwards the partner's scans onto the page the customer is already watching.

We will thread the telecom support desk through the rest of the piece. The child agent owns account checks. The parent agent is the desk's front door.

## The architecture: two agents, three MCP servers, one event loop

The full stack in the [companion repo]([REPO-LINK]) is a parent Streamlit app, a parent Deep Agent, a parent MCP server, a server-side child Deep Agent, and a child MCP server. Here we keep every piece except Streamlit, and print the events a UI would render instead.

The flow looks like this:

```
user ─▶ parent Deep Agent ─▶ parent MCP: run_downstream_react_agent
                                  └─▶ child Deep Agent ─▶ child MCP: validate_email_address
                                                                     give_profile_information
```

The library choices are deliberate:

- **FastMCP 3.4.8** for both servers. Its `Context.report_progress()` sends progress notifications over Streamable HTTP.
- **langchain-mcp-adapters 0.3.2** for the client side. Its `Callbacks(on_progress=...)` hook is where MCP progress enters the LangChain world.
- **deepagents 0.7.19** for both agents, with the filesystem tools and general-purpose subagent switched off, so each agent sees only its MCP tools.
- **langchain-google-genai 4.4.0** with Gemini 2.5 Flash (`gemini-2.5-flash`), temperature 0, with thought summaries turned on so the UI can show reasoning.

One shortcut up front. All three servers run as background tasks inside the notebook's event loop. In the repo they are separate processes, which is what you want in real life: a crash in one server should not take down the others.

You need a `GEMINI_API_KEY` in your environment, and a full run of this notebook costs a few cents of Gemini 2.5 Flash.

In [1]:
# !uv pip install deepagents==0.7.19 fastmcp==3.4.8 langchain-google-genai==4.4.0 langchain-mcp-adapters==0.3.2
import asyncio
import json
import logging
import os
import re
import time
from contextvars import ContextVar
from uuid import uuid4

from deepagents import (
    GeneralPurposeSubagentProfile,
    HarnessProfile,
    create_deep_agent,
    register_harness_profile,
)
from fastmcp import Client, Context, FastMCP
from langchain_core.callbacks import adispatch_custom_event
from langchain_core.runnables import ensure_config
from langchain_core.tools import StructuredTool
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_mcp_adapters.callbacks import Callbacks
from langchain_mcp_adapters.client import MultiServerMCPClient

if not os.environ.get("GEMINI_API_KEY"):
    raise RuntimeError("GEMINI_API_KEY is not set. Export it before starting Jupyter.")

logging.getLogger("langchain_google_genai").setLevel(logging.ERROR)  # silence schema-key warnings

MODEL = "gemini-2.5-flash"
STEP_DELAY_SECONDS = 2.0  # simulated latency per tool stage
CHILD_URL = "http://127.0.0.1:8001/mcp"
NAIVE_PARENT_URL = "http://127.0.0.1:8000/mcp"
STREAMING_PARENT_URL = "http://127.0.0.1:8002/mcp"
QUERY = (
    "Please validate tuhin@gmail.com and then show me the profile information "
    "for that email address. Do the same for alice@gmail.com."
)

`STEP_DELAY_SECONDS` is the only knob that changes the story. At 2 seconds a full notebook run takes about 2.5 minutes; the repo's 5 seconds makes the silence more painful and the run slower.

## Step 1: a child server whose tools take their time

The child tools stand in for real account systems: a mail-domain check, a CRM lookup. Each one does three stages of slow work and reports progress after each stage with `ctx.report_progress()`.

Each tool also keeps an `execution_trace` of its stages and returns it in the result. That matters later, for clients that never render progress.

Let's write the two child tools.

In [2]:
child_mcp = FastMCP("child-mcp")
BLOCKED_EMAIL_DOMAINS = {"invalid.example", "blocked.example"}
DEMO_PROFILES = {"tuhin@gmail.com": {"display_name": "Tuhin", "plan": "developer",
                                     "enabled_preferences": ["product_updates"]}}
EMAIL_PATTERN = re.compile(r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9-]+(?:\.[A-Za-z0-9-]+)+$")


async def report_step(ctx: Context, trace: list, event: str, message: str) -> None:
    # simulate slow async work, then push one MCP progress notification
    await asyncio.sleep(STEP_DELAY_SECONDS)
    trace.append({"sequence": len(trace) + 1, "event": event, "message": message})
    await ctx.report_progress(progress=len(trace), total=3, message=message)


@child_mcp.tool()
async def validate_email_address(ctx: Context, email_address: str) -> dict:
    """Validate an email address in three streamed stages."""
    email, trace = email_address.strip().lower(), []
    format_ok = bool(EMAIL_PATTERN.fullmatch(email))
    await report_step(ctx, trace, "format_checked",
                      f"{email}: format is {'valid' if format_ok else 'invalid'} (1 of 3)")
    domain = email.rsplit("@", 1)[-1]
    domain_ok = format_ok and domain not in BLOCKED_EMAIL_DOMAINS
    await report_step(ctx, trace, "domain_checked",
                      f"{email}: domain {domain} is {'allowed' if domain_ok else 'blocked'} (2 of 3)")
    policy_ok = domain_ok and not email.startswith(("noreply", "no-reply"))
    await report_step(ctx, trace, "delivery_policy_checked",
                      f"{email}: delivery policy {'passed' if policy_ok else 'failed'} (3 of 3)")
    return {"status": "completed", "tool": "validate_email_address", "execution_trace": trace,
            "result": {"email_address": email, "is_valid": policy_ok}}


@child_mcp.tool()
async def give_profile_information(ctx: Context, email_address: str) -> dict:
    """Look up a demo account profile in three streamed stages."""
    email, trace = email_address.strip().lower(), []
    profile = DEMO_PROFILES.get(email)
    await report_step(ctx, trace, "account_record_checked",
                      f"{email}: account record {'found' if profile else 'not found'} (1 of 3)")
    await report_step(ctx, trace, "preferences_loaded",
                      f"{email}: preferences {'loaded' if profile else 'skipped'} (2 of 3)")
    await report_step(ctx, trace, "profile_response_prepared",
                      f"{email}: response prepared (3 of 3)")
    return {"status": "completed", "tool": "give_profile_information", "execution_trace": trace,
            "result": {"email_address": email, "profile_found": bool(profile), "profile": profile}}

The tools are simulated. `asyncio.sleep()` replaces real I/O, which keeps the demo deterministic, but it also hides the messy parts of real backends: timeouts, partial failures, and stages that do not come in neat thirds.

Let's start the child server on port 8001 and ask it for its tool list.

In [3]:
SERVER_TASKS = {}


async def start_server(mcp: FastMCP, url: str) -> None:
    # run a Streamable HTTP MCP server as a background task on the notebook's event loop
    port = int(url.rsplit(":", 1)[1].split("/")[0])
    SERVER_TASKS[mcp.name] = asyncio.create_task(mcp.run_http_async(
        show_banner=False, transport="streamable-http",
        host="127.0.0.1", port=port, path="/mcp", log_level="error"))
    await asyncio.sleep(1.0)
    async with Client(url) as client:
        names = [tool.name for tool in await client.list_tools()]
    print(f"{mcp.name} listening on {url} with tools: {names}")


await start_server(child_mcp, CHILD_URL)

child-mcp listening on http://127.0.0.1:8001/mcp with tools: ['validate_email_address', 'give_profile_information']


The server is up and advertises both tools over Streamable HTTP. Nothing has called them yet.

## Step 2: one Deep Agents runner for both agents

Both agents, child and parent, need the same thing: run a Deep Agent against one MCP server and turn everything that happens into a flat stream of UI events. The repo puts this in `deep_agent_factory.py`, and we reuse it for both levels.

Two LangChain pieces do the work. `adispatch_custom_event()` lets any code inside a run push a named event into that run's [`astream_events()`](https://python.langchain.com/docs/how_to/callbacks_custom_events/) stream. `ContextVar`s remember which run and which tool call is active, so a progress callback that knows nothing about LangChain can still land its event in the right place.

Let's write the progress callback and the tool wrapper.

In [4]:
TOOL_STREAM_EVENT, TOOL_LIFECYCLE_EVENT = "tool_stream", "tool_lifecycle"
RELAY_PREFIX = "streaming-mcp:"
_active_run_config = ContextVar("active_run_config", default=None)
_active_call_id = ContextVar("active_call_id", default=None)


def decode_relay_event(message):
    """Return the relayed child event, or None for an ordinary progress message."""
    if not message or not message.startswith(RELAY_PREFIX):
        return None
    try:
        payload = json.loads(message.removeprefix(RELAY_PREFIX))
    except json.JSONDecodeError:
        return None
    return payload if isinstance(payload, dict) else None


async def on_progress(progress, total, message, context) -> None:
    # MCP progress notification -> LangChain custom event on the agent's own stream
    relay = decode_relay_event(message)
    payload = {
        "tool": context.tool_name or "mcp_tool",
        "call_id": _active_call_id.get() or "unknown",
        "chunk": relay.get("text", "") if relay else message or "",
        "source": "child_agent" if relay else "mcp",
        "stream_kind": relay.get("kind", "mcp_progress") if relay else "mcp_progress",
        "child_tool": relay.get("tool") if relay else None,
    }
    config = _active_run_config.get()
    if config is not None:
        await adispatch_custom_event(TOOL_STREAM_EVENT, payload, config=config)


def wrap_tool(base_tool) -> StructuredTool:
    """Wrap an MCP tool so each call gets an id and emits start/end/error events."""
    async def call(**kwargs):
        config = ensure_config()
        call_id = str(uuid4())
        config_token = _active_run_config.set(config)
        call_token = _active_call_id.set(call_id)
        event = {"tool": base_tool.name, "call_id": call_id}
        try:
            await adispatch_custom_event(TOOL_LIFECYCLE_EVENT, {"phase": "start", **event}, config=config)
            result = await base_tool.ainvoke(kwargs)
            await adispatch_custom_event(TOOL_LIFECYCLE_EVENT, {"phase": "end", **event}, config=config)
            return result
        except Exception:
            await adispatch_custom_event(TOOL_LIFECYCLE_EVENT, {"phase": "error", **event}, config=config)
            raise
        finally:
            _active_call_id.reset(call_token)
            _active_run_config.reset(config_token)

    return StructuredTool(name=base_tool.name, description=base_tool.description,
                          args_schema=base_tool.args_schema, coroutine=call)

`wrap_tool()` gives every call its own `call_id` and brackets it with start and end events. That id is what lets a UI keep two concurrent calls to the same tool in separate cards. `on_progress()` already knows how to unpack a relayed child event; for now it will only ever see plain messages.

Next, the agent itself. `astream_events()` yields many event types; we keep four: our two custom events, model text, and model reasoning blocks.

In [5]:
# keep the Deep Agents harness but expose only the MCP tools (no filesystem, no subagent)
register_harness_profile(f"google_genai:{MODEL}", HarnessProfile(
    excluded_tools=frozenset({"ls", "read_file", "write_file", "edit_file", "delete", "glob", "grep", "execute"}),
    excluded_middleware=frozenset({"SummarizationMiddleware"}),
    general_purpose_subagent=GeneralPurposeSubagentProfile(enabled=False),
))


def build_process_query(*, server_name: str, server_url: str, system_prompt: str):
    """Return an async generator that runs one Deep Agent against one MCP server."""
    async def process_query(query: str):
        client = MultiServerMCPClient(
            {server_name: {"transport": "streamable_http", "url": server_url,
                           "timeout": 600, "sse_read_timeout": 600}},
            callbacks=Callbacks(on_progress=on_progress))
        tools = [wrap_tool(tool) for tool in await client.get_tools()]
        model = ChatGoogleGenerativeAI(model=MODEL, temperature=0, max_tokens=4096,
                                       streaming=True, include_thoughts=True)
        agent = create_deep_agent(model=model, tools=tools, system_prompt=system_prompt, subagents=[])

        async for event in agent.astream_events({"messages": [{"role": "user", "content": query}]},
                                                version="v2"):
            if event["event"] == "on_custom_event" and event["name"] == TOOL_STREAM_EVENT:
                yield {"type": "on_tool_stream", **event["data"]}
            elif event["event"] == "on_custom_event" and event["name"] == TOOL_LIFECYCLE_EVENT:
                yield {"type": f"on_tool_{event['data']['phase']}", **event["data"]}
            elif event["event"] == "on_chat_model_stream":
                chunk = event["data"]["chunk"]
                for block in chunk.content_blocks:
                    if block.get("type") == "reasoning" and block.get("reasoning"):
                        yield {"type": "on_agent_thinking", "text": block["reasoning"]}
                if chunk.text:
                    yield {"type": "on_chat_model_stream", "text": chunk.text}

    return process_query

The repo renders these events in Streamlit as expandable cards per tool call. In a notebook we print them with a timestamp instead, and measure two numbers that decide how the experience feels: the longest silence between visible events, and how long the user waits for the first tool detail.

In [6]:
async def watch(process_query, query: str, label: str) -> dict:
    """Print what a chat UI would show, with timestamps, and measure the silences."""
    start = last = time.perf_counter()
    gaps, counts, answer, first_detail = [], {}, [], None
    async for event in process_query(query):
        now = time.perf_counter()
        kind = event["type"]
        counts[kind] = counts.get(kind, 0) + 1
        if kind == "on_chat_model_stream":
            answer.append(event["text"])
            continue  # tokens are collected and printed once at the end
        gaps.append(now - last)
        last = now
        if kind == "on_tool_stream" and first_detail is None:
            first_detail = now - start
        tool = event.get("tool", "")
        if event.get("child_tool"):
            tool = f"{tool} > {event['child_tool']}"
        text = event.get("chunk") or event.get("text") or ""
        text = " ".join(text.split())[:80]
        shown = event["stream_kind"] if event.get("source") == "child_agent" else kind
        print(f"{now - start:5.1f}s  {shown:<19} {tool:<46} {text}")
    total = time.perf_counter() - start
    print(f"\nanswer: {' '.join(''.join(answer).split())[:300]}")
    stats = {"run": label, "total_s": round(total, 1), "events_shown": len(gaps),
             "tool_stream_events": counts.get("on_tool_stream", 0),
             "thinking_events": counts.get("on_agent_thinking", 0),
             "longest_silence_s": round(max(gaps), 1) if gaps else None,
             "first_tool_detail_s": round(first_detail, 1) if first_detail else None}
    print(stats)
    return stats

## Step 3: the child agent streams fine on its own

Before nesting anything, let's confirm the child agent streams when we talk to it directly.

In [7]:
child_agent = build_process_query(
    server_name="child-mcp",
    server_url=CHILD_URL,
    system_prompt=(
        "You are a child agent for account support. Use validate_email_address for "
        "verification and give_profile_information for profile questions. If the user asks "
        "for both, validate first. Finish with a concise summary without repeating live updates."
    ),
)
child_stats = await watch(child_agent, QUERY, "child agent, direct")

  2.0s  on_agent_thinking                                                  **Sequencing the Tasks** I'm now thinking about the workflow: I'll tackle the em


  2.6s  on_agent_thinking                                                  **Defining the Procedure** I've decided to proceed step-by-step. First, I will v
  2.7s  on_tool_start       validate_email_address                         


  4.7s  on_tool_stream      validate_email_address                         tuhin@gmail.com: format is valid (1 of 3)


  6.7s  on_tool_stream      validate_email_address                         tuhin@gmail.com: domain gmail.com is allowed (2 of 3)


  8.7s  on_tool_stream      validate_email_address                         tuhin@gmail.com: delivery policy passed (3 of 3)
  8.7s  on_tool_end         validate_email_address                         


...

 28.1s  on_tool_stream      give_profile_information                       alice@gmail.com: account record not found (1 of 3)


 30.1s  on_tool_stream      give_profile_information                       alice@gmail.com: preferences skipped (2 of 3)


 32.1s  on_tool_stream      give_profile_information                       alice@gmail.com: response prepared (3 of 3)
 32.1s  on_tool_end         give_profile_information                       


 33.9s  on_agent_thinking                                                  **Confirming Profile Retrieval** I've successfully retrieved the profile informa


 34.6s  on_agent_thinking                                                  **Summarizing Completed Profiles** I've finished retrieving both profiles. For T

answer: tuhin@gmail.com was validated and a profile was found with display name Tuhin, developer plan, and product updates enabled. alice@gmail.com was validated, but no profile was found.
{'run': 'child agent, direct', 'total_s': 34.6, 'events_shown': 27, 'tool_stream_events': 12, 'thinking_events': 7, 'longest_silence_s': 2.1, 'first_tool_detail_s': 4.7}


This is what a good experience looks like. The child took 34.6 seconds end to end, but the screen showed 27 events: 7 reasoning summaries and 12 tool progress updates among them. The longest silence was 2.1 seconds, and the first tool detail arrived at 4.7 seconds.

Notice the order. The agent validated `tuhin@gmail.com`, fetched its profile, then did the same for `alice@gmail.com`, exactly as asked. Alice validates but has no profile, which is also correct for our demo data.

## Step 4: the naive wrapper goes dark

Now the support desk wants this child behind its front-door agent. The obvious move is to wrap the child agent in an MCP tool: run it, collect its final answer, return it.

Let's build that naive parent server on port 8000.

In [8]:
naive_parent_mcp = FastMCP("parent-mcp-naive")


@naive_parent_mcp.tool()
async def run_downstream_react_agent(query: str) -> dict:
    """Run the account-support child agent and return its final answer."""
    final = []
    async for event in child_agent(query):
        if event["type"] == "on_chat_model_stream":
            final.append(event["text"])
    return {"status": "completed", "final_result": "".join(final).strip()}


await start_server(naive_parent_mcp, NAIVE_PARENT_URL)

parent-mcp-naive listening on http://127.0.0.1:8000/mcp with tools: ['run_downstream_react_agent']


The tool body consumes the child's full event stream and throws away everything except the answer tokens. The parent agent only ever receives the result.

Let's point a parent Deep Agent at it and run the same request.

In [9]:
PARENT_PROMPT = (
    "You are the parent agent. Always call run_downstream_react_agent with the user's full "
    "request, then give a concise summary without repeating the live updates."
)
naive_parent = build_process_query(server_name="parent-mcp", server_url=NAIVE_PARENT_URL,
                                   system_prompt=PARENT_PROMPT)
naive_stats = await watch(naive_parent, QUERY, "parent, naive MCP wrapper")

  1.8s  on_agent_thinking                                                  **Delegating the Task** I've realized the need to validate email addresses and r


  2.3s  on_agent_thinking                                                  **Refining the Approach** I'm now zeroing in on calling the `run_downstream_reac
  2.4s  on_tool_start       run_downstream_react_agent                     


 37.0s  on_tool_end         run_downstream_react_agent                     



answer: tuhin@gmail.com was validated and its profile information was retrieved, showing Tuhin as the display name, a developer plan, and product updates enabled. alice@gmail.com was validated, but no profile information was found.
{'run': 'parent, naive MCP wrapper', 'total_s': 37.9, 'events_shown': 4, 'tool_stream_events': 0, 'thinking_events': 2, 'longest_silence_s': 34.6, 'first_tool_detail_s': None}


The final answer is just as good. The experience is not. The screen showed 4 events in 37.9 seconds, and between `on_tool_start` at 2.4 seconds and `on_tool_end` at 37.0 seconds there was nothing: a 34.6-second silence with 0 tool updates.

That is not good enough. The child produced the same reasoning and the same 12 progress updates as before; the wrapper simply had nowhere to send them. On the repo's 5-second stages, that blank stretches past a minute.

## Step 5: smuggle the child's stream through `message`

So, how do we get those events across? The MCP progress notification carries `progress`, an optional `total`, and a `message` string. There is no slot for structured data, so we put structured data in the string.

The envelope is a fixed prefix plus compact JSON. The prefix matters: a client can tell a relayed child event from an ordinary progress message like "50% done" without guessing.

Let's write the encoder and round-trip one event.

In [10]:
def encode_relay_event(*, kind: str, text: str, tool=None, call_id=None) -> str:
    """Pack one child-agent event into the string-only MCP progress message field."""
    payload = {"kind": kind, "text": text}
    if tool:
        payload["tool"] = tool
    if call_id:
        payload["call_id"] = call_id
    return RELAY_PREFIX + json.dumps(payload, separators=(",", ":"))


sample = encode_relay_event(kind="child_mcp_progress", text="tuhin@gmail.com: format is valid (1 of 3)",
                            tool="validate_email_address", call_id="c-42")
print(sample)
print(decode_relay_event(sample))
print(decode_relay_event("50% done"))

streaming-mcp:{"kind":"child_mcp_progress","text":"tuhin@gmail.com: format is valid (1 of 3)","tool":"validate_email_address","call_id":"c-42"}
{'kind': 'child_mcp_progress', 'text': 'tuhin@gmail.com: format is valid (1 of 3)', 'tool': 'validate_email_address', 'call_id': 'c-42'}
None


The relayed event decodes back to its kind, text, child tool name and call id. The plain "50% done" string decodes to `None`, so ordinary MCP servers keep working with the same client code.

Now the relay itself. It is the naive tool with one change: instead of dropping child events, it forwards each one through `ctx.report_progress()` with the envelope as the message. Child tool starts, child MCP progress, child reasoning, and child answer tokens all get their own `kind`.

The tool also builds an `execution_trace`, skipping reasoning and answer tokens to keep it compact, and returns it next to `final_result`.

Let's build the streaming parent server on port 8002.

In [11]:
streaming_parent_mcp = FastMCP("parent-mcp-streaming")
RELAY_KINDS = {"on_tool_start": "child_tool_started", "on_tool_stream": "child_mcp_progress",
               "on_tool_end": "child_tool_finished", "on_tool_error": "child_tool_failed",
               "on_agent_thinking": "child_reasoning", "on_chat_model_stream": "child_response"}


@streaming_parent_mcp.tool()
async def run_downstream_react_agent(ctx: Context, query: str) -> dict:
    """Run the account-support child agent, relaying every child event as MCP progress."""
    step, trace, final = 0, [], []

    async def emit(kind: str, text: str, tool=None, call_id=None) -> None:
        nonlocal step
        step += 1
        if kind not in {"child_reasoning", "child_response"}:  # keep the replay trace compact
            trace.append({"sequence": len(trace) + 1, "event": kind, "message": text, "tool": tool})
        await ctx.report_progress(progress=step, total=None,
                                  message=encode_relay_event(kind=kind, text=text, tool=tool, call_id=call_id))

    await emit("child_started", "Server-side child agent started")
    async for event in child_agent(query):
        kind = event["type"]
        if kind == "on_chat_model_stream":
            final.append(event["text"])
        text = {"on_tool_start": f"Calling {event.get('tool')}",
                "on_tool_end": f"Finished {event.get('tool')}",
                "on_tool_error": f"{event.get('tool')} failed"}.get(kind) or event.get("chunk") or event.get("text", "")
        await emit(RELAY_KINDS[kind], text, tool=event.get("tool"), call_id=event.get("call_id"))
    await emit("child_completed", "Server-side child agent completed")
    return {"status": "completed", "execution_trace": trace,
            "final_result": "".join(final).strip() or "Child workflow completed."}


await start_server(streaming_parent_mcp, STREAMING_PARENT_URL)

parent-mcp-streaming listening on http://127.0.0.1:8002/mcp with tools: ['run_downstream_react_agent']


The parent side needs no new code. The same `build_process_query()` runner already decodes relayed events in `on_progress()`, and `watch()` prints the child's `kind` and tool name for them.

Let's run the same request through the streaming parent.

In [12]:
streaming_parent = build_process_query(server_name="parent-mcp", server_url=STREAMING_PARENT_URL,
                                       system_prompt=PARENT_PROMPT)
streaming_stats = await watch(streaming_parent, QUERY, "parent, streaming relay")

  2.1s  on_agent_thinking                                                  **Validating the Request** I'm focusing on the user's intent: validating email a
  2.1s  on_tool_start       run_downstream_react_agent                     
  2.2s  child_started       run_downstream_react_agent                     Server-side child agent started


  3.9s  child_reasoning     run_downstream_react_agent                     **Validating and Extracting Profiles** I'm focusing on the sequential processing


  4.6s  child_reasoning     run_downstream_react_agent                     **Sequencing Email Operations** I'm now prioritizing a sequential workflow. Firs
  4.7s  child_tool_started  run_downstream_react_agent > validate_email_address Calling validate_email_address


  6.7s  child_mcp_progress  run_downstream_react_agent > validate_email_address tuhin@gmail.com: format is valid (1 of 3)


  8.7s  child_mcp_progress  run_downstream_react_agent > validate_email_address tuhin@gmail.com: domain gmail.com is allowed (2 of 3)


...

 31.8s  child_mcp_progress  run_downstream_react_agent > give_profile_information alice@gmail.com: preferences skipped (2 of 3)


 33.8s  child_mcp_progress  run_downstream_react_agent > give_profile_information alice@gmail.com: response prepared (3 of 3)
 33.8s  child_tool_finished run_downstream_react_agent > give_profile_information Finished give_profile_information


 35.3s  child_reasoning     run_downstream_react_agent                     **Validating Account Data** I have completed the validation phase for the provid


 36.1s  child_reasoning     run_downstream_react_agent                     **Summarizing Request Fulfillment** I've now finalized summarizing the actions t
 36.1s  child_response      run_downstream_react_agent                     tuhin@gmail.com was validated and a profile was found for
 36.1s  child_response      run_downstream_react_agent                     Tuhin, who has a developer plan with product updates enabled. alice@gmail.com wa
 36.1s  child_completed     run_downstream_react_agent                     Server-side child agent completed
 36.2s  on_tool_end         run_downstream_react_agent                     



answer: tuhin@gmail.com was validated and a profile was found for Tuhin, who has a developer plan with product updates enabled. alice@gmail.com was validated, but no profile information was found.
{'run': 'parent, streaming relay', 'total_s': 37.1, 'events_shown': 34, 'tool_stream_events': 31, 'thinking_events': 1, 'longest_silence_s': 2.1, 'first_tool_detail_s': 2.2}


The parent's screen now tells the whole story. In 37.1 seconds it showed 34 events, 31 of them relayed from inside the tool call: the child's start, 7 reasoning summaries, 4 child tool starts, 12 child MCP progress updates, 4 child tool finishes, 2 child answer chunks, and the child's completion. The longest silence dropped from 34.6 seconds to 2.1 seconds, and the first tool detail arrived at 2.2 seconds.

The UI can also tell nested calls apart. Each relayed event carries the child tool name and its own call id, so `run_downstream_react_agent > validate_email_address` renders as a card inside the parent's card. That is what the repo's Streamlit app does.

One cost is visible here too. The parent agent wrote its own summary after the child had already streamed one, so the answer appears twice in spirit. The parent prompt asks it not to repeat the live updates; it mostly complies, but a production UI should decide which answer is authoritative.

## Step 6: the numbers side by side

Let's put the three runs in one table.

In [13]:
rows = [child_stats, naive_stats, streaming_stats]
columns = ["run", "total_s", "events_shown", "tool_stream_events", "thinking_events",
           "longest_silence_s", "first_tool_detail_s"]
print(f"{'':<26}" + "".join(f"{c:>20}" for c in columns[1:]))
for row in rows:
    print(f"{row['run']:<26}" + "".join(f"{str(row[c]):>20}" for c in columns[1:]))

                                       total_s        events_shown  tool_stream_events     thinking_events   longest_silence_s first_tool_detail_s
child agent, direct                       34.6                  27                  12                   7                 2.1                 4.7
parent, naive MCP wrapper                 37.9                   4                   0                   2                34.6                None
parent, streaming relay                   37.1                  34                  31                   1                 2.1                 2.2


Total time barely moved: 34.6, 37.9 and 37.1 seconds. The relay does not make the work faster. It changes what the user sees while the work happens: from 4 events and a 34.6-second silence to 34 events and no gap longer than 2.1 seconds.

These figures come from one run. Gemini output is not bit-reproducible even at temperature 0, so the reasoning counts and model-turn timings shift a little between runs. The tool timings are fixed by `STEP_DELAY_SECONDS`.

## Clients that ignore progress still get the story

Not every MCP client renders progress notifications. Some, Cursor among them when we built this, show only the final tool result. For those, the relay's `execution_trace` is the fallback: the same events, in order, inside the result.

Let's call the streaming tool with a plain FastMCP client that registers no progress handler.

In [14]:
# a client that ignores progress notifications (Cursor today) still gets the full story
async with Client(STREAMING_PARENT_URL) as client:
    result = await client.call_tool("run_downstream_react_agent", {"query": QUERY})
payload = result.structured_content
print(f"status: {payload['status']}, trace entries: {len(payload['execution_trace'])}")
for entry in payload["execution_trace"][:6]:
    print(f"  {entry['sequence']:>2}. {entry['event']:<20} {entry['message'][:70]}")
print(f"  ...\nfinal_result: {' '.join(payload['final_result'].split())[:200]}")

status: completed, trace entries: 22
   1. child_started        Server-side child agent started
   2. child_tool_started   Calling validate_email_address
   3. child_mcp_progress   tuhin@gmail.com: format is valid (1 of 3)
   4. child_mcp_progress   tuhin@gmail.com: domain gmail.com is allowed (2 of 3)
   5. child_mcp_progress   tuhin@gmail.com: delivery policy passed (3 of 3)
   6. child_tool_finished  Finished validate_email_address
  ...
final_result: tuhin@gmail.com was validated and a profile was found with display name Tuhin, developer plan, and product updates enabled. alice@gmail.com was validated, but no profile was found.


The client saw no live updates, but the result carries 22 trace entries: the child's start, four tool calls with a start, three stages and a finish each, and the child's completion. That is 1 + 4 × 5 + 1 = 22. It is a replay, not a stream, but the user can at least see what happened and in what order.

## Running the full stack from the repo

The notebook squeezes everything into one process. The [companion repo]([REPO-LINK]) runs the real thing: two MCP servers and two Streamlit apps, each in its own terminal. The README has the same steps; here is the short version.

**1. Install.** The project uses [uv](https://docs.astral.sh/uv/) and Python 3.11 or newer. From the repo root:

```bash
git clone [REPO-LINK] streaming-mcp
cd streaming-mcp
uv sync
```

**2. Configure.** Copy the example environment file and put your Gemini key in it. The defaults give the parent MCP server port 8000 and the child MCP server port 8001, both on `127.0.0.1` at path `/mcp`.

```bash
cp .env.example .env
# then edit .env and set GEMINI_API_KEY=<your key>
```

`GEMINI_MODEL` is optional and defaults to `gemini-2.5-flash`. Each child tool stage takes 5 seconds by default. Set `CHILD_MCP_STEP_DELAY_SECONDS=0.1` in `.env` when you want a fast smoke test instead of a realistic wait.

**3. Bring up the MCP servers.** Start the child first, because the parent's tool calls it. Use one terminal each:

```bash
# terminal 1: child MCP server (validate_email_address, give_profile_information) on :8001
uv run streaming-mcp-child-server
```

```bash
# terminal 2: parent MCP server (run_downstream_react_agent) on :8000
uv run streaming-mcp-parent-server
```

**4. Bring up the Streamlit apps.** The child app talks to the child agent directly. The parent app goes through the full nested path.

```bash
# terminal 3: child UI at http://localhost:8501
uv run streamlit run child_streamlit_app.py --server.port 8501
```

```bash
# terminal 4: parent UI at http://localhost:8502
uv run streamlit run parent_streamlit_app.py --server.port 8502
```

**5. Try it.** Paste the same request into both apps:

```text
Please validate tuhin@gmail.com and then show me the profile information for that email address. do the same for alice@gmail.com.
```

The child app shows one expandable card per tool call, filling in three stages each. The parent app shows one `run_downstream_react_agent` card that fills with the child agent's lifecycle, reasoning, both tools' streams and the child's summary, then the parent's own answer.

**6. Run the tests.** The unit tests cover the relay envelope and the `execution_trace` replay, and they need neither a Gemini key nor running servers:

```bash
uv run python -m unittest tests.test_events
```

To try the parent tool from an MCP client such as Cursor, point the client at `http://127.0.0.1:8000/mcp` while terminals 1 and 2 are running. Expect the final result with its `execution_trace` rather than live updates.

## From demo to production

Start with what is wrong with what we just built.

**Everything shares one event loop.** We ran three servers and two agents in one process to keep the notebook self-contained. A slow child would starve the parent's server, and one crash ends everything. Run each MCP server as its own process, as the repo does with `streaming-mcp-parent-server` and `streaming-mcp-child-server`.

**The runner rebuilds its client and agent per query.** `build_process_query()` opens a new `MultiServerMCPClient` and calls `create_deep_agent()` on every request. That is fine for a demo and wasteful under load. Cache the tool list and the compiled agent, and reconnect only on failure.

**The child's reasoning leaves the building.** We relay every thought summary to the parent's user. For an internal support desk that is useful. For a customer-facing parent, or a child that reasons over account data, it is a data-leak path. Decide per deployment which `kind`s cross the boundary, and drop `child_reasoning` by default when the audiences differ.

**The envelope is a convention, not a standard.** Any client that does not know the `streaming-mcp:` prefix sees raw JSON in its progress bar. Keep the `text` field human-readable for that reason, and version the prefix (`streaming-mcp/v1:`) before a second team depends on it.

**There is no backpressure.** One parent call produced 31 notifications here; a chattier child with token-level answer streaming can produce hundreds. The repo's Streamlit bridge caps its queue at 200 pending events. Batch answer tokens on the server side, or relay them at sentence boundaries.

**Failures need to stream too.** The wrapper emits `child_tool_failed` when a child tool raises, but a child agent that hangs will hold the parent tool open until the 600-second timeout. Add a heartbeat and a deadline on the parent tool, and surface both as relay events.

**Evals belong on the stream, not only on the answer.** The `execution_trace` is a ready-made eval artifact: assert the tool order, the number of stages, and that validation precedes profile lookup. That catches a child agent that skips a step but still writes a confident summary.

**The model choice moves cost and latency.** Gemini 2.5 Flash with thought summaries added roughly 1.5 to 2.5 seconds per model turn here, and the child made five turns. A lighter model shortens those gaps and cuts cost; a larger model plans better on messy requests at a higher price per turn. Thought summaries add output tokens, so switch them off where nobody reads them.

The hard limit is that MCP progress is one-way and best-effort: the relay can show the user everything the child does, but the parent agent itself still reasons only over the final result.

## The bottom line

- Wrapping a streaming agent in an MCP tool silences it: 34.6 seconds of nothing in our run.
- MCP progress notifications are enough to carry a whole agent's event stream, if you pack structured events into the `message` string behind a prefix.
- A relay changed the longest silence from 34.6 to 2.1 seconds without changing total time.
- Return an `execution_trace` too, for clients that never render progress.

## Resources

- Companion repo: [REPO-LINK]
- MCP specification, progress notifications: https://modelcontextprotocol.io/specification/2025-06-18/basic/utilities/progress
- Deep Agents: https://github.com/langchain-ai/deepagents
- FastMCP progress reporting: https://gofastmcp.com/servers/progress
- LangChain MCP adapters: https://github.com/langchain-ai/langchain-mcp-adapters
- LangChain custom stream events: https://python.langchain.com/docs/how_to/callbacks_custom_events/